In [31]:
# Cell 1 - imports & config

import os
import uuid
import json
import base64
from pathlib import Path
import csv
from datetime import datetime

import numpy as np
import pandas as pd

# === Paths ===
BASE_DIR = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output")

MODULE_INF_FILES = {
    "battery":      BASE_DIR / "battery_inference.csv",
    "body":         BASE_DIR / "body_inference.csv",
    "engine":       BASE_DIR / "engine_inference.csv",
    "transmission": BASE_DIR / "transmission_inference.csv",
    "tyre":         BASE_DIR / "tyre_inference.csv",
}

OUT_CSV  = BASE_DIR / "vehicle_alerts.csv"
OUT_XLSX = BASE_DIR / "vehicle_alerts.xlsx"   # optional; can be commented out

# === Expected common columns in *_inference.csv ===
COMMON_NON_FEATURE_COLS = [
    "row_hash",
    "timestamp",
    "date",
    "source_id",
    "kafka_key",
    "offset",
    "source_file",
    "recon_error_dense",
    "recon_error_lstm",
    "lstm_window_id",
    "isolation_score",
    "kde_logp",
    "gmm_logp",
    "composite_score",
    "anomaly_label",
    "anomaly_severity",
    "inference_run_id",
    "inference_timestamp",
    "processing_latency_ms",
    "dense_per_feature_error_json",
    "explain_top_k_json",
    "raw_model_outputs_json",
]

# === Alert policy knobs for refinement ===
# We *refine* the model's anomaly_severity using composite_score distribution.
# Within the anomaly population (anomaly_severity > 0):
#   0–PCT_ANOM          => stays normal (no alert)
#   PCT_ANOM–PCT_WARN   => severity 1 ("anomaly")
#   PCT_WARN–PCT_CRIT   => severity 2 ("warning")
#   PCT_CRIT–1.0        => severity 3 ("critical")
PCT_ANOM = 0.50
PCT_WARN = 0.85
PCT_CRIT = 0.90

# GAP in seconds for merging point alerts into windows
GAP_S = 60  # 5 minutes


In [32]:
# Cell 2 - helpers

def to_b64_json_safe(obj):
    """
    Take obj (dict/list/str/number) and return a base64-encoded JSON string.
    - If obj is a string that *already* looks like base64 JSON and decodes cleanly, return as-is.
    - Otherwise, encode JSON and then base64.
    This keeps all complex structures safe for CSV (no commas, quotes, or newlines).
    """
    if obj is None:
        return ""

    if isinstance(obj, str):
        s = obj.strip()
        if not s:
            return ""
        try:
            # heuristic: base64 chars only, length multiple of 4
            if all(c in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=" for c in s) and len(s) % 4 == 0:
                decoded = base64.b64decode(s)
                try:
                    json.loads(decoded.decode("utf-8"))
                    return s
                except Exception:
                    pass
        except Exception:
            pass

        # not an existing base64 JSON string -> try to parse as JSON
        try:
            parsed = json.loads(s)
            raw = json.dumps(parsed, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
            return base64.b64encode(raw).decode("ascii")
        except Exception:
            # treat as plain string; wrap as JSON string then base64
            raw = json.dumps(s, ensure_ascii=False).encode("utf-8")
            return base64.b64encode(raw).decode("ascii")

    # For dict/list/number/etc – JSON encode then base64
    try:
        raw = json.dumps(obj, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
        return base64.b64encode(raw).decode("ascii")
    except Exception:
        # last-resort fallback: str() then base64
        raw = str(obj).encode("utf-8")
        return base64.b64encode(raw).decode("ascii")


def short_alert_id():
    """Short but reasonably unique id; 16 hex chars."""
    return uuid.uuid4().hex[:16]


def safe_dt(x):
    """
    Accept strings or pandas timestamps. Return ISO 8601 string or empty.
    We don't enforce timezone; we just keep whatever is there.
    """
    if pd.isna(x):
        return ""
    if isinstance(x, (pd.Timestamp, datetime)):
        try:
            return x.isoformat()
        except Exception:
            return str(x)
    try:
        return pd.to_datetime(x).isoformat()
    except Exception:
        return str(x)


def severity_label_from_int(sev: int) -> str:
    """
    0 = normal
    1 = anomaly        (baseline anomaly band)
    2 = warning        (stronger anomaly band)
    3 = critical       (most extreme anomaly band)
    """
    if sev >= 3:
        return "critical"
    elif sev == 2:
        return "warning"
    elif sev == 1:
        return "anomaly"
    return "normal"


def ensure_columns(df: pd.DataFrame, cols):
    """Ensure all columns in cols exist in df (filled with empty string if missing)."""
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df


In [33]:
# Cell 3 - read module CSVs defensively and stack

dfs = {}
missing = []

for module, p in MODULE_INF_FILES.items():
    if not p.exists():
        missing.append(module)
        continue

    # Read as strings to preserve base64 and timestamps exactly
    df = pd.read_csv(
        p,
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python"
    )
    df.columns = [c.strip() for c in df.columns]  # normalize whitespace in col names

    # ensure expected non-feature columns are present
    df = ensure_columns(df, COMMON_NON_FEATURE_COLS)

    # Tag module
    df["__module"] = module

    dfs[module] = df

if missing:
    raise FileNotFoundError(f"Missing module inference files for: {missing}")

# Stack into one frame
all_df = pd.concat(dfs.values(), ignore_index=True, sort=False)

print("Per-module row counts:", {m: len(df) for m, df in dfs.items()})
print("Total stacked rows:", len(all_df))

# Ensure composite_score is numeric (for refinement)
all_df["composite_score"] = pd.to_numeric(all_df["composite_score"], errors="coerce")

# anomaly_severity is numeric 0/1/2/3 from the model
all_df["anomaly_severity"] = pd.to_numeric(all_df["anomaly_severity"], errors="coerce").fillna(0).astype(int)


Per-module row counts: {'battery': 31799, 'body': 31799, 'engine': 31799, 'transmission': 31799, 'tyre': 31799}
Total stacked rows: 158995


In [34]:
# Cell 4 - severity refinement using percentile bands

valid_comp = all_df["composite_score"].dropna()
if len(valid_comp) == 0:
    raise ValueError("No valid composite_score values found; cannot compute thresholds.")

# We prefer to refine ONLY points that model already marked as anomalous (severity > 0)
anom_mask = (all_df["anomaly_severity"] > 0) & all_df["composite_score"].notna()
anom_scores = all_df.loc[anom_mask, "composite_score"]

if len(anom_scores) > 0:
    base_scores = anom_scores
    gating_mask = anom_mask
    print(f"Using {len(anom_scores)} model-anomaly points (anomaly_severity > 0) for refinement.")
else:
    # If model is too "normal" or misconfigured, fall back to all scores
    base_scores = valid_comp
    gating_mask = all_df["composite_score"].notna()
    print("WARNING: No anomaly_severity > 0 points found.")
    print("Falling back to severity refinement based on composite_score over ALL points.")

# Compute quantile thresholds on the chosen base population
q_anom = base_scores.quantile(PCT_ANOM)   # boundary between normal and anomaly band
q_warn = base_scores.quantile(PCT_WARN)   # anomaly -> warning
q_crit = base_scores.quantile(PCT_CRIT)   # warning -> critical

# Guard against degenerate distributions
if not (q_anom < q_warn < q_crit):
    max_comp = base_scores.max()
    min_comp = base_scores.min()
    # fallback: evenly spaced thresholds between min and max
    q_anom = min_comp + PCT_ANOM * (max_comp - min_comp)
    q_warn = min_comp + PCT_WARN * (max_comp - min_comp)
    q_crit = min_comp + PCT_CRIT * (max_comp - min_comp)

print("Refinement thresholds:")
print(f"  ANOMALY threshold (q{int(PCT_ANOM*100)}): {q_anom:.6f}")
print(f"  WARNING threshold (q{int(PCT_WARN*100)}): {q_warn:.6f}")
print(f"  CRITICAL threshold (q{int(PCT_CRIT*100)}): {q_crit:.6f}")

score_series = all_df["composite_score"]

# Start everything as normal (0)
all_df["alert_severity"] = 0

# Only points in gating_mask participate in refinement
# Banding logic:
#   < q_anom            => 0 (normal)
#   [q_anom, q_warn)    => 1 (anomaly)
#   [q_warn, q_crit)    => 2 (warning)
#   >= q_crit           => 3 (critical)

# severity 1: anomaly band
anom_band = gating_mask & (score_series >= q_anom)
all_df.loc[anom_band, "alert_severity"] = 1

# severity 2: warning band (overrides anomaly band)
warn_band = gating_mask & (score_series >= q_warn)
all_df.loc[warn_band, "alert_severity"] = 2

# severity 3: critical band (overrides warning band)
crit_band = gating_mask & (score_series >= q_crit)
all_df.loc[crit_band, "alert_severity"] = 3

# Derive labels
all_df["alert_severity_label"] = all_df["alert_severity"].apply(severity_label_from_int)

print("Point-level severity value counts:")
print(all_df["alert_severity"].value_counts().sort_index())
print("Point-level severity label counts:")
print(all_df["alert_severity_label"].value_counts())


Using 158995 model-anomaly points (anomaly_severity > 0) for refinement.
Refinement thresholds:
  ANOMALY threshold (q50): 0.483582
  WARNING threshold (q85): 0.608616
  CRITICAL threshold (q90): 0.715256
Point-level severity value counts:
alert_severity
0    79497
1    55648
2     7950
3    15900
Name: count, dtype: int64
Point-level severity label counts:
alert_severity_label
normal      79497
anomaly     55648
critical    15900
warning      7950
Name: count, dtype: int64


In [35]:
# Cell 5 - candidate alerts (one row per inference row with alert_severity > 0)

# Filter out normal points
cand_df = all_df[all_df["alert_severity"] > 0].copy()

print("Candidate point alerts:", len(cand_df))

def derive_candidate_row(row):
    # vehicle_id fallback logic
    vehicle_id = row.get("source_id") or row.get("kafka_key") or row.get("vehicle_id") or ""

    ts_raw = row.get("timestamp") or row.get("inference_timestamp") or ""
    ts_iso = safe_dt(ts_raw)
    date_val = row.get("date") or (pd.to_datetime(ts_iso).date().isoformat() if ts_iso else "")

    # top features -> base64
    top_features_b64 = to_b64_json_safe(row.get("explain_top_k_json", ""))

    row_hash = row.get("row_hash") or ""

    # composite score and severity
    comp = row.get("composite_score")
    try:
        comp_float = float(comp)
    except Exception:
        comp_float = np.nan

    sev = int(row.get("alert_severity", 0))
    sev_lbl = severity_label_from_int(sev)

    return {
        "alert_id": short_alert_id(),
        "vehicle_id": vehicle_id,
        "module": row["__module"],
        "start_ts": ts_iso,
        "end_ts": ts_iso,
        "duration_s": 0,
        "n_points": 1,
        "peak_composite": comp_float,
        "mean_composite": comp_float,
        "severity": sev,
        "severity_label": sev_lbl,
        "top_features_b64": top_features_b64,
        "row_hash": row_hash,
        "date": date_val,
    }

cand_rows = [derive_candidate_row(r) for _, r in cand_df.iterrows()]
cand_df2 = pd.DataFrame(cand_rows)

print("Candidate alerts after normalization:", len(cand_df2))


Candidate point alerts: 79498
Candidate alerts after normalization: 79498


In [36]:
# Cell 6 - merge adjacent point alerts into per-module windows
# - One row per (alert window, module, vehicle_id)
# - top_features_b64 and row_hash taken from the *peak* composite_score point in the window
# - Window severity determined by voting over point severities, not max()

WARN_FRAC = 0.40   # promote to warning if >= 40% of points are sev>=2
CRIT_FRAC = 0.40   # promote to critical if >= 40% of points are sev==3

if cand_df2.empty:
    print("No candidate alerts; no windows to create.")
    merged_df = cand_df2.copy()
else:
    # Sort by vehicle, module, timestamp
    cand_df2 = cand_df2.sort_values(["vehicle_id", "module", "start_ts"]).reset_index(drop=True)

    # Convert to datetime for gap calculation
    def to_ts(x):
        try:
            return pd.to_datetime(x)
        except Exception:
            return pd.NaT

    cand_df2["_ts"] = cand_df2["start_ts"].apply(to_ts)

    merged = []

    def finalize_window(cur):
        """
        Finalize severity for a window based on point-level severity counts.
        Mutates cur in-place and returns it.
        """
        total_n = int(cur.get("n_points", 1))
        if total_n <= 0:
            total_n = 1  # safety

        sev1_cnt = int(cur.get("_sev1_cnt", 0))
        sev2_cnt = int(cur.get("_sev2_cnt", 0))
        sev3_cnt = int(cur.get("_sev3_cnt", 0))

        frac3 = sev3_cnt / total_n
        frac2plus3 = (sev2_cnt + sev3_cnt) / total_n

        if frac3 >= CRIT_FRAC:
            final_sev = 3
        elif frac2plus3 >= WARN_FRAC:
            final_sev = 2
        else:
            # We know this window has at least one alert point (cand_df2 is sev>0),
            # so if it doesn't qualify for warning/critical, treat as anomaly.
            final_sev = 1

        cur["severity"] = final_sev
        cur["severity_label"] = severity_label_from_int(final_sev)
        return cur

    for (veh, mod), g in cand_df2.groupby(["vehicle_id", "module"], sort=False):
        g = g.sort_values("_ts").reset_index(drop=True)
        if g.empty:
            continue

        # Initialize first window from row 0
        cur = g.loc[0].to_dict()
        cur["_start"] = cur["_ts"]
        cur["_end"] = cur["_ts"]

        # init severity counts for this window
        sev0 = int(cur.get("severity", 0))
        cur["_sev1_cnt"] = 1 if sev0 == 1 else 0
        cur["_sev2_cnt"] = 1 if sev0 == 2 else 0
        cur["_sev3_cnt"] = 1 if sev0 == 3 else 0

        for i in range(1, len(g)):
            row = g.loc[i].to_dict()
            row_ts = row.get("_ts", pd.NaT)

            if pd.isna(row_ts) or pd.isna(cur["_end"]):
                # can't compare times: close current window and start a new one
                merged.append(finalize_window(cur))
                cur = row
                cur["_start"] = cur["_ts"]
                cur["_end"] = cur["_ts"]

                sev0 = int(cur.get("severity", 0))
                cur["_sev1_cnt"] = 1 if sev0 == 1 else 0
                cur["_sev2_cnt"] = 1 if sev0 == 2 else 0
                cur["_sev3_cnt"] = 1 if sev0 == 3 else 0
                continue

            gap = (row_ts - cur["_end"]).total_seconds()
            if gap <= GAP_S:
                # Merge into current window
                cur["_end"] = row_ts

                # update n_points
                n_cur = int(cur.get("n_points", 1))
                n_row = int(row.get("n_points", 1))
                total_n = n_cur + n_row

                # update mean_composite (weighted)
                mc_cur = float(cur.get("mean_composite", np.nan))
                mc_row = float(row.get("mean_composite", np.nan))
                if np.isnan(mc_cur):
                    new_mean = mc_row
                elif np.isnan(mc_row):
                    new_mean = mc_cur
                else:
                    new_mean = (mc_cur * n_cur + mc_row * n_row) / max(1, total_n)

                cur["n_points"] = total_n
                cur["mean_composite"] = new_mean

                # update peak_composite and representative row_hash + top_features
                pc_cur = float(cur.get("peak_composite", -np.inf))
                pc_row = float(row.get("peak_composite", -np.inf))
                if pc_row >= pc_cur:
                    cur["peak_composite"] = pc_row
                    cur["top_features_b64"] = row.get("top_features_b64", "") or ""
                    cur["row_hash"] = row.get("row_hash", "") or ""

                # update severity counts (do NOT update cur["severity"] here)
                s = int(row.get("severity", 0))
                cur["_sev1_cnt"] += 1 if s == 1 else 0
                cur["_sev2_cnt"] += 1 if s == 2 else 0
                cur["_sev3_cnt"] += 1 if s == 3 else 0

            else:
                # gap too large: close current window, start a new one
                merged.append(finalize_window(cur))
                cur = row
                cur["_start"] = cur["_ts"]
                cur["_end"] = cur["_ts"]

                sev0 = int(cur.get("severity", 0))
                cur["_sev1_cnt"] = 1 if sev0 == 1 else 0
                cur["_sev2_cnt"] = 1 if sev0 == 2 else 0
                cur["_sev3_cnt"] = 1 if sev0 == 3 else 0

        # don't forget last window
        merged.append(finalize_window(cur))

    merged_df = pd.DataFrame(merged).reset_index(drop=True)

    # Drop internal helper columns (severity counts) – not needed downstream
    for col in ["_sev1_cnt", "_sev2_cnt", "_sev3_cnt"]:
        if col in merged_df.columns:
            merged_df = merged_df.drop(columns=[col])

    print("Merged alert windows:", len(merged_df))


Merged alert windows: 99


In [37]:
# Cell 7 - finalize columns and ensure clean schema for CSV
# No dense errors, no linked_row_hashes; only top_features_b64 + one row_hash per alert

if merged_df.empty:
    final_df = pd.DataFrame(columns=[
        "alert_id",
        "vehicle_id",
        "module",
        "start_ts",
        "end_ts",
        "duration_s",
        "n_points",
        "peak_composite",
        "mean_composite",
        "severity",
        "severity_label",
        "top_features_b64",
        "date",
        "row_hash",
    ])
else:
    # Compute start/end timestamps and durations as ISO + seconds
    merged_df["start_ts"] = merged_df["_start"].apply(lambda x: x.isoformat() if not pd.isna(x) else "")
    merged_df["end_ts"]   = merged_df["_end"].apply(lambda x: x.isoformat() if not pd.isna(x) else "")

    merged_df["duration_s"] = (merged_df["_end"] - merged_df["_start"]).dt.total_seconds().fillna(0).astype(int)

    # Ensure numeric types are sane
    merged_df["n_points"] = merged_df["n_points"].astype(int)
    merged_df["peak_composite"] = pd.to_numeric(merged_df["peak_composite"], errors="coerce")
    merged_df["mean_composite"] = pd.to_numeric(merged_df["mean_composite"], errors="coerce")
    merged_df["severity"] = merged_df["severity"].astype(int)

    # Make sure all text columns are strings – very important for CSV safety
    text_cols = [
        "alert_id",
        "vehicle_id",
        "module",
        "start_ts",
        "end_ts",
        "severity_label",
        "top_features_b64",
        "date",
        "row_hash",
    ]
    for c in text_cols:
        if c in merged_df.columns:
            merged_df[c] = merged_df[c].astype(str).fillna("")
        else:
            merged_df[c] = ""

    # Define final column order
    final_cols = [
        "alert_id",
        "vehicle_id",
        "module",
        "start_ts",
        "end_ts",
        "duration_s",
        "n_points",
        "peak_composite",
        "mean_composite",
        "severity",
        "severity_label",
        "top_features_b64",
        "date",
        "row_hash",
    ]

    final_df = merged_df[final_cols].copy()

print("Final alerts schema:")
print(final_df.dtypes)
print("Final alert rows:", len(final_df))


Final alerts schema:
alert_id             object
vehicle_id           object
module               object
start_ts             object
end_ts               object
duration_s            int64
n_points              int64
peak_composite      float64
mean_composite      float64
severity              int64
severity_label       object
top_features_b64     object
date                 object
row_hash             object
dtype: object
Final alert rows: 99


In [38]:
# Cell 8 - write robust CSV and validate structure

if final_df.empty:
    print("No alerts to write. Skipping file creation.")
else:
    # Write CSV with robust settings
    final_df.to_csv(
        OUT_CSV,
        index=False,
        encoding="utf-8",
        quoting=csv.QUOTE_MINIMAL,
        lineterminator="\n"  # Excel is fine with \n
    )
    print(f"Wrote CSV: {OUT_CSV}")

    # OPTIONAL: also write an Excel version
    try:
        final_df.to_excel(OUT_XLSX, index=False)
        print(f"Wrote Excel: {OUT_XLSX}")
    except Exception as e:
        print(f"Could not write XLSX (non-fatal): {e}")

    # === Validation step 1: check consistent column count per row using csv.reader ===
    issues = []
    with OUT_CSV.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)

    if not rows:
        print("CSV seems empty after writing. This should not happen if final_df was non-empty.")
    else:
        header = rows[0]
        expected_cols = len(header)
        for i, row in enumerate(rows[1:], start=2):  # 1-based line numbers, skip header
            if len(row) != expected_cols:
                issues.append((i, len(row), row[:5]))  # store first few fields for debug

    if issues:
        print("CSV validation FAILED. Rows with incorrect column counts:")
        for line_no, col_count, snippet in issues[:10]:
            print(f"  Line {line_no}: {col_count} columns, snippet={snippet}")
        print(f"Total problematic rows: {len(issues)}")
    else:
        print("CSV validation PASSED: all rows have the correct number of columns.")

    # === Validation step 2: sanity check with pandas read_csv ===
    try:
        df_check = pd.read_csv(OUT_CSV, engine="python")
        if list(df_check.columns) != list(final_df.columns):
            print("WARNING: Column names changed when re-reading CSV with pandas.")
            print("  Written columns:", list(final_df.columns))
            print("  Read-back columns:", list(df_check.columns))
        if len(df_check) != len(final_df):
            print(f"WARNING: Row count mismatch after re-read: written={len(final_df)}, read={len(df_check)}")
        else:
            print("Pandas re-read sanity check OK: row count matches.")
    except Exception as e:
        print(f"WARNING: pandas could not read back the CSV cleanly: {e}")


Wrote CSV: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\vehicle_alerts.csv
Could not write XLSX (non-fatal): No module named 'openpyxl'
CSV validation PASSED: all rows have the correct number of columns.
Pandas re-read sanity check OK: row count matches.


In [39]:
anom_mask = (all_df["anomaly_severity"] > 0) & all_df["composite_score"].notna()
if anom_mask.sum() > 0:
    base_scores = all_df.loc[anom_mask, "composite_score"]
else:
    base_scores = all_df["composite_score"].dropna()

q_anom_val = base_scores.quantile(PCT_ANOM)
q_warn_val = base_scores.quantile(PCT_WARN)
q_crit_val = base_scores.quantile(PCT_CRIT)

print("\n================ COMPOSITE SCORE THRESHOLD VALUES ================")
print(f"Percentile {PCT_ANOM:0.2f} → anomaly threshold:   {q_anom_val:.6f}")
print(f"Percentile {PCT_WARN:0.2f} → warning threshold:   {q_warn_val:.6f}")
print(f"Percentile {PCT_CRIT:0.2f} → critical threshold:  {q_crit_val:.6f}")

print("\n=== Distribution summary ===")
print(base_scores.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())



================ COMPOSITE SCORE THRESHOLD VALUES ================
Percentile 0.50 → anomaly threshold:   0.483582
Percentile 0.85 → warning threshold:   0.608616
Percentile 0.90 → critical threshold:  0.715256

=== Distribution summary ===
count    158995.000000
mean          0.508705
std           0.137025
min           0.335814
50%           0.483582
75%           0.565084
90%           0.715256
95%           0.832554
99%           0.913228
max           0.914853


In [40]:
# Cell X — Alert Count Summary (point-level + window-level)

print("\n======================= ALERT COUNT SUMMARY =======================")

# 1) Point-level after refinement (cand_df2)
print("\n--- Point-level Alerts (cand_df2) ---")
if "cand_df2" in globals() and not cand_df2.empty:
    pt_counts = cand_df2["severity"].value_counts().sort_index()
    for sev in [1, 2, 3]:
        cnt = pt_counts.get(sev, 0)
        label = severity_label_from_int(sev)
        print(f"{label:<10}: {cnt}")
else:
    print("cand_df2 is empty or missing.")


# 2) Window-level counts (merged_df)
print("\n--- Final Window-level Alerts (merged_df) ---")
if "merged_df" in globals() and not merged_df.empty:
    win_counts = merged_df["severity"].value_counts().sort_index()
    for sev in [1, 2, 3]:
        cnt = win_counts.get(sev, 0)
        label = severity_label_from_int(sev)
        print(f"{label:<10}: {cnt}")
else:
    print("merged_df is empty or missing.")

print("\n===================================================================")



======================= ALERT COUNT SUMMARY =======================

--- Point-level Alerts (cand_df2) ---
anomaly   : 55648
critical  : 15900

--- Final Window-level Alerts (merged_df) ---
anomaly   : 91
critical  : 4



In [41]:
# Cell — Count alerts above warning & critical thresholds

# Fixed thresholds
ANOMALY_THR  = 0.483582
WARNING_THR  = 0.608616
CRITICAL_THR = 0.832554

if "final_df" not in globals() or final_df.empty:
    raise ValueError("final_df is missing or empty. Ensure final_df is loaded.")

df = final_df.copy()
df["peak_composite"] = pd.to_numeric(df["peak_composite"], errors="coerce")

# Drop NaN peak values
df = df.dropna(subset=["peak_composite"])

# Counts
above_critical = df[df["peak_composite"] >= CRITICAL_THR].shape[0]
above_warning  = df[df["peak_composite"] >= WARNING_THR].shape[0]
between_warn_crit = df[(df["peak_composite"] >= WARNING_THR) & (df["peak_composite"] < CRITICAL_THR)].shape[0]
below_warning = df[df["peak_composite"] < WARNING_THR].shape[0]

print("\n================ PEAK COMPOSITE THRESHOLD COUNTS ================")
print(f"Rows ≥ CRITICAL ({CRITICAL_THR}):        {above_critical}")
print(f"Rows ≥ WARNING  ({WARNING_THR}):        {above_warning}")
print(f"Rows between WARNING and CRITICAL:      {between_warn_crit}")
print(f"Rows < WARNING threshold:               {below_warning}")
print("==================================================================\n")



================ PEAK COMPOSITE THRESHOLD COUNTS ================
Rows ≥ CRITICAL (0.832554):        13
Rows ≥ WARNING  (0.608616):        52
Rows between WARNING and CRITICAL:      39
Rows < WARNING threshold:               47

